# Context-Aware FAQ Retrieval for Student Helpdesks
## Google Colab implementation — four levels

**Model:** Sentence Transformer `all-MiniLM-L6-v2` with 384-dimensional embeddings and cosine similarity. This notebook uses the same model specified in the project architecture; it does **not** use TF-IDF.

**Levels**
1. Data collection — load three FAQ datasets (1,500 FAQ records total) and labelled test queries.
2. Preprocessing — normalize text consistently using the same cleaning function for FAQ questions and user queries.
3. Sentence Transformer — encode FAQ questions using `all-MiniLM-L6-v2` into normalized 384-dimensional embeddings and save the FAQ index.
4. Retrieval and evaluation — encode each user query, rank FAQ records using cosine similarity, apply threshold `0.42`, return stored answers, and calculate Top-1, Top-3, and MRR.

This is a retrieval system: it returns answers stored in the dataset and does not generate new answers. The included FAQ data is demonstration data and should be validated before presenting it as official college information.


## Setup: install libraries and upload the project ZIP
Download and extract the project ZIP on your computer only if you want to inspect its files. In this notebook, click Run on the next cell and upload `CampusConnect_FAQ_Helpdesk_MiniLM_1500.zip`.

In [ ]:
%pip -q install sentence-transformers pandas numpy scikit-learn


In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Choose context_aware_faq_full_project.zip
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Please upload the project ZIP file.")
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")
PROJECT_DIR = "/content/context_aware_faq_full_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
print("Project folder:", PROJECT_DIR)
print("Dataset files:", os.listdir(DATA_DIR))


## Level 1 — Data collection
Load and combine the three FAQ CSV datasets and load the labelled test-query CSV. The project package contains 500 FAQs in each of the three datasets (1,500 total) and 120 test queries.


In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(PROJECT_DIR) / "data"
faq_paths = sorted(DATA_PATH.glob("faq_dataset_*.csv"))
if len(faq_paths) != 3:
    raise FileNotFoundError(f"Expected 3 FAQ datasets, found {len(faq_paths)}: {faq_paths}")
faqs = pd.concat([pd.read_csv(p).fillna("") for p in faq_paths], ignore_index=True)
test_path = DATA_PATH / "test_queries.csv"
tests = pd.read_csv(test_path).fillna("")
required_faq_cols = {"faq_id", "question", "answer", "category"}
if not required_faq_cols.issubset(faqs.columns):
    raise ValueError(f"FAQ files must contain columns: {sorted(required_faq_cols)}")
if not {"query", "expected_faq_id"}.issubset(tests.columns):
    raise ValueError("test_queries.csv must contain query and expected_faq_id columns")

print("LEVEL 1: DATA COLLECTION")
print("Datasets:", [p.name for p in faq_paths])
print("FAQ records:", len(faqs))
print("Test queries:", len(tests))
print("FAQ columns:", list(faqs.columns))
display(faqs.head(10))
print("FAQ count by dataset:")
for p in faq_paths:
    print(p.name, len(pd.read_csv(p)))
print("Sample test queries:")
display(tests.head(10))


## Level 2 — Text preprocessing
Normalize case, punctuation, and extra spaces. Stopwords are not removed because words such as “not” can affect meaning.


In [ ]:
import re

def clean_text(text):
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\brevaluation\b", "re evaluation", text)
    text = re.sub(r"\badmit card\b", "hall ticket", text)
    text = re.sub(r"\bstudent id\b", "college id", text)
    if "attendance" in text:
        text += " minimum required eligibility percentage"
    return re.sub(r"\s+", " ", text).strip()

faqs["cleaned_question"] = faqs["question"].apply(clean_text)
tests["cleaned_query"] = tests["query"].apply(clean_text)

print("LEVEL 2: PREPROCESSING")
display(faqs[["question", "cleaned_question"]].head(10))
display(tests[["query", "cleaned_query"]].head(10))


## Level 3 — Sentence Transformer model: MiniLM L6 v2
`all-MiniLM-L6-v2` encodes each cleaned FAQ question as a 384-dimensional semantic embedding. The embeddings are L2-normalized and saved as the FAQ index. The same model and cleaning function are used for user queries.


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)
faq_questions = faqs["cleaned_question"].astype(str).tolist()
faq_answers = faqs["answer"].astype(str).tolist()
faq_ids = faqs["faq_id"].astype(int).tolist()
faq_categories = faqs["category"].astype(str).tolist()
faq_embeddings = model.encode(
    faq_questions,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32
).astype(np.float32)

index_dir = Path(PROJECT_DIR) / "saved_index"
index_dir.mkdir(exist_ok=True)
np.save(index_dir / "faq_embeddings.npy", faq_embeddings)
faqs[["faq_id", "question", "answer", "category"]].to_csv(index_dir / "faq_index_metadata.csv", index=False)

print("LEVEL 3: SENTENCE TRANSFORMER EMBEDDINGS")
print("Model:", MODEL_NAME)
print("Embedding matrix shape:", faq_embeddings.shape)
print("Embedding dimensions:", faq_embeddings.shape[1])
print("Saved FAQ index:", index_dir)


## Level 4 — Retrieval, answer selection, and evaluation
Encode a user's question, compare it with all FAQ vectors using cosine similarity, and rank the closest matches. If the best score is below the threshold, the system declines to return a potentially unrelated answer. The threshold is a starting value and should be tuned with representative validation queries.

In [ ]:
THRESHOLD = 0.42  # specified by the project architecture; validate using labelled queries

def retrieve_answer(query, top_k=3, threshold=THRESHOLD):
    cleaned_query = clean_text(query)
    if not cleaned_query:
        return {"found": False, "message": "Please enter a question.", "best_score": None, "results": []}
    query_embedding = model.encode(
        [cleaned_query], convert_to_numpy=True, normalize_embeddings=True
    )[0].astype(np.float32)
    # With normalized embeddings, dot product is equivalent to cosine similarity.
    scores = faq_embeddings @ query_embedding
    k = max(1, min(int(top_k), len(faqs)))
    order = np.argsort(scores)[::-1][:k]
    results = []
    for idx in order:
        results.append({
            "faq_id": int(faq_ids[idx]),
            "question": str(faq_questions[idx]),
            "answer": str(faq_answers[idx]),
            "category": str(faq_categories[idx]),
            "score": float(scores[idx]),
        })
    best = results[0]
    found = best["score"] >= float(threshold)
    message = best["answer"] if found else "I couldn't find a close enough match in the FAQ dataset. Please rephrase your question or contact the relevant college office."
    return {"found": found, "message": message, "best_score": best["score"], "results": results}


### Try the FAQ system
Enter a student-helpdesk question. Example: `How can I download my hall ticket?`

In [ ]:
query = input("Enter your question: ")
result = retrieve_answer(query)

print("\\nAnswer:", result["message"])
if result["best_score"] is not None:
    print(f"Similarity score: {result['best_score']:.3f}")
print("\\nTop matches:")
for item in result["results"]:
    print(f"FAQ {item['faq_id']} | score={item['score']:.3f} | {item['question']}")


### Evaluate the model
Top-1 accuracy measures how often the correct FAQ ranks first. Top-3 accuracy measures whether it appears in the first three results. MRR rewards higher ranking positions. These are results on the supplied labelled sample test set, not a guarantee of real-world performance.

In [ ]:
top1_hits = 0
top3_hits = 0
reciprocal_ranks = []
evaluation_rows = []

for _, test in tests.iterrows():
    expected_id = int(test["expected_faq_id"])
    result = retrieve_answer(test["cleaned_query"], top_k=3, threshold=0.0)
    ranked_ids = [item["faq_id"] for item in result["results"]]
    rank = ranked_ids.index(expected_id) + 1 if expected_id in ranked_ids else None
    top1_hits += int(rank == 1)
    top3_hits += int(rank is not None)
    reciprocal_ranks.append(1 / rank if rank else 0)
    evaluation_rows.append({
        "query": test["query"],
        "expected_faq_id": expected_id,
        "top_result_faq_id": ranked_ids[0],
        "correct_rank": rank
    })

n = len(tests)
print("LEVEL 4: EVALUATION RESULTS")
print("Number of test queries:", n)
print(f"Top-1 accuracy: {top1_hits / n:.2%}")
print(f"Top-3 accuracy: {top3_hits / n:.2%}")
print(f"Mean Reciprocal Rank (MRR): {sum(reciprocal_ranks) / n:.3f}")
display(pd.DataFrame(evaluation_rows).head(20))


## How to explain the four levels during your presentation
- **Level 1 — Data collection:** We combine three FAQ datasets containing 1,500 FAQ records and load 120 labelled test queries.
- **Level 2 — Preprocessing:** We lowercase text, normalize selected terms, remove punctuation, and collapse extra spaces. The same function is applied to FAQ questions and user queries.
- **Level 3 — Sentence Transformer:** `all-MiniLM-L6-v2` converts FAQ questions into normalized 384-dimensional semantic embeddings. The saved index contains embeddings and FAQ metadata.
- **Level 4 — Retrieval and evaluation:** The user query is embedded using the same model. Cosine similarity ranks the stored FAQs; the system returns the best stored answer only when the top score meets threshold `0.42`. Top-1, Top-3, and MRR are evaluated on the labelled test queries.

**Important:** The threshold `0.42` follows the architecture diagram but should be validated using representative labelled queries. The sample dataset is demonstration data, not verified official college policy.
